# Chapter 6 – Hypothesis Testing: Python Exercises

**Data Analysis for Business, Economics, and Policy** · Békés & Kézdi  
**Dataset**: `retail_sales_data.csv` — Retail Store Sales Campaign

---

## How this notebook works

| Part | What you do |
|------|-------------|
| **Worked Example** | Fully solved — read and run every cell, understand each step |
| **Your Turn** | Same structure, different angle — you write the code |

**The 4-step pattern** used in every exercise:
1. **State H₀ and Hₐ** — write them as comments before any code
2. **Descriptive statistics** — means, std devs, sample sizes
3. **Run the t-test** — compute t and p-value
4. **Decide and interpret** — is |t| > 2? What does p tell you in plain English?

> **Slide map** — refer back to these while working:  
> Slides 2–3 · What is a hypothesis · Sales context example  
> Slides 5–6 · t-statistic formula and intuition  
> Slides 11–13 · Worked examples (same dataset)  
> Slides 14–15 · Dataset overview and exercise roadmap


## Setup — run this cell first

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from itertools import combinations
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ── Load the dataset ──────────────────────────────────────────────────────────
df = pd.read_csv('retail_sales_data.csv')

print(f"Shape: {df.shape}  ({df['store_id'].nunique()} stores × {df['week'].nunique()} weeks)")
print(f"\nColumns: {list(df.columns)}")
print(f"\nPeriods: {df['period'].value_counts().to_dict()}")
print(f"Regions: {sorted(df['region'].unique())}")
print(f"Store sizes: {sorted(df['store_size'].unique())}")
print(f"\nWeekly revenue summary:")
df['weekly_revenue_k'].describe().round(2)

In [ ]:
# ── Quick data preview ───────────────────────────────────────────────────────
print("First 8 rows (one store, all weeks):")
df[df['store_id'] == 1].head(8)

---
# Exercise 1 — One-Sample t-test

**Business question**: Before the campaign, was average weekly revenue already above the industry benchmark of £25k per store?

> **Refer to Slide 11 (Worked Example A)** for the step-by-step solution this mirrors.  
> **Formula**: `t = (x-bar − target) / SE`  where `SE = std / √n`


## Worked Example 1 — Is pre-campaign revenue above £25k/week?

In [ ]:
# ── WORKED EXAMPLE 1 ────────────────────────────────────────────────────────
# STEP 1: State hypotheses
# H₀: μ_revenue = 25  (pre-campaign mean equals the £25k benchmark)
# Hₐ: μ_revenue ≠ 25  (two-sided: could be above or below)
# α = 0.05

null_benchmark = 25

pre_revenue = df[df['period'] == 'Pre']['weekly_revenue_k']

# STEP 2: Descriptive statistics
mean_pre = pre_revenue.mean()
std_pre  = pre_revenue.std()
n_pre    = len(pre_revenue)
se_pre   = std_pre / np.sqrt(n_pre)

print("─── Step 2: Descriptives ───────────────────────────────")
print(f"  Pre-campaign mean:  £{mean_pre:.2f}k/week")
print(f"  Std deviation:      £{std_pre:.2f}k")
print(f"  n (store-weeks):    {n_pre}")
print(f"  Standard Error:     £{se_pre:.3f}k")

# STEP 3: Run the t-test
t_stat, p_value = stats.ttest_1samp(pre_revenue, popmean=null_benchmark)

print(f"\n─── Step 3: t-test ─────────────────────────────────────")
print(f"  t = ({mean_pre:.2f} − {null_benchmark}) / {se_pre:.3f} = {t_stat:.3f}")
print(f"  p-value: {p_value:.4f}")
print(f"  |t| > 2? {abs(t_stat) > 2}")

# STEP 4: Decision and interpretation
print(f"\n─── Step 4: Decision ───────────────────────────────────")
if p_value < 0.05:
    print(f"  p = {p_value:.4f} < 0.05  →  REJECT H₀")
    print(f"  Pre-campaign revenue (£{mean_pre:.1f}k) is significantly")
    print(f"  above the £{null_benchmark}k industry benchmark.")
else:
    print(f"  p ≥ 0.05  →  Do NOT reject H₀")

In [ ]:
# ── Visualisation: Worked Example 1 ─────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.hist(pre_revenue, bins=30, color='steelblue', alpha=0.7, edgecolor='white')
ax1.axvline(mean_pre,        color='#1A3A5C', lw=2.5, label=f'Pre mean = £{mean_pre:.1f}k')
ax1.axvline(null_benchmark,  color='#C0392B', lw=2.5, ls='--', label=f'Benchmark = £{null_benchmark}k')
ax1.set_xlabel('Weekly revenue (£k)'); ax1.set_ylabel('Count')
ax1.set_title('Pre-campaign revenue distribution'); ax1.legend()

x  = np.linspace(-10, 10, 500)
y  = stats.norm.pdf(x)
cv = 2.0
ax2.plot(x, y, '#1A3A5C', lw=2)
ax2.fill_between(x, y, where=(x > cv),  color='#C0392B', alpha=0.3, label='Rejection region')
ax2.fill_between(x, y, where=(x < -cv), color='#C0392B', alpha=0.3)
ax2.axvline(t_stat, color='#D97706', lw=3, label=f't = {t_stat:.2f}')
ax2.axvline( cv, color='#C0392B', lw=1, ls=':')
ax2.axvline(-cv, color='#C0392B', lw=1, ls=':', label=f'±{cv}')
ax2.set_xlabel('t-statistic'); ax2.set_title('t-distribution: our t is well in the tail')
ax2.legend(fontsize=9); ax2.set_xlim(-10, 10)

plt.suptitle('Worked Example 1: Pre-campaign revenue vs £25k benchmark', fontsize=13)
plt.tight_layout(); plt.show()

## Your Turn 1 — Is post-campaign revenue above £25k/week?

Repeat the same test but for the **post-campaign** period.  
Does the conclusion change? Why might that be expected?

**Refer to Slide 11** for the 4-step structure.


In [ ]:
# ── YOUR TURN 1 ─────────────────────────────────────────────────────────────
# STEP 1: State hypotheses
# H₀: ?
# Hₐ: ?
# α = 0.05

null_benchmark = 25
post_revenue = df[df['period'] == 'Post']['weekly_revenue_k']

# STEP 2: Descriptive statistics
# YOUR CODE HERE

# STEP 3: Run the t-test
# t_stat_yt1, p_value_yt1 = stats.ttest_1samp(...)

# STEP 4: Decision and interpretation
# YOUR CODE HERE


*Interpretation: ...*

> ...

---
# Exercise 2 — Two-Sample t-test

**Business question**: Did the email campaign significantly increase weekly revenue across stores?

> **Refer to Slide 12 (Worked Example B)** for the step-by-step solution this mirrors.  
> **Formula**: `t = (x-bar_post − x-bar_pre) / SE_diff`


## Worked Example 2 — Post-campaign vs pre-campaign revenue

In [ ]:
# ── WORKED EXAMPLE 2 ────────────────────────────────────────────────────────
# STEP 1: State hypotheses
# H₀: μ_post = μ_pre  (campaign had no effect on revenue)
# Hₐ: μ_post ≠ μ_pre  (two-sided)
# α = 0.05

pre  = df[df['period'] == 'Pre']['weekly_revenue_k']
post = df[df['period'] == 'Post']['weekly_revenue_k']

# STEP 2: Descriptive statistics
diff = post.mean() - pre.mean()
se_diff = np.sqrt(pre.var()/len(pre) + post.var()/len(post))

print("─── Step 2: Descriptives ───────────────────────────────")
print(f"  Pre-campaign:  mean=£{pre.mean():.2f}k, std=£{pre.std():.2f}k, n={len(pre)}")
print(f"  Post-campaign: mean=£{post.mean():.2f}k, std=£{post.std():.2f}k, n={len(post)}")
print(f"  Difference:    £{diff:.2f}k/week")
print(f"  SE of diff:    £{se_diff:.3f}k")

# STEP 3: Run the t-test
t_stat2, p_value2 = stats.ttest_ind(post, pre, equal_var=False)

print(f"\n─── Step 3: t-test ─────────────────────────────────────")
print(f"  t = {t_stat2:.3f}")
print(f"  p-value: {p_value2:.6f}")
print(f"  |t| > 2? {abs(t_stat2) > 2}")

# STEP 4: Decision
print(f"\n─── Step 4: Decision ───────────────────────────────────")
if p_value2 < 0.05:
    print(f"  REJECT H₀  (p < 0.001)")
    print(f"  Post-campaign revenue is £{diff:.2f}k/week higher on average.")
    print(f"  Strong evidence the campaign increased revenue.")

In [ ]:
# ── Visualisation: Worked Example 2 ─────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.hist(pre,  bins=28, alpha=0.6, color='#D97706', label=f'Pre  (n={len(pre)})', edgecolor='white')
ax1.hist(post, bins=28, alpha=0.6, color='#0D7377', label=f'Post (n={len(post)})', edgecolor='white')
ax1.axvline(pre.mean(),  color='#D97706', lw=2.5, ls='--')
ax1.axvline(post.mean(), color='#0D7377', lw=2.5, ls='--')
ax1.set_xlabel('Weekly revenue (£k)'); ax1.set_ylabel('Count')
ax1.set_title('Pre vs Post-campaign revenue'); ax1.legend()

bp = ax2.boxplot([pre, post], labels=['Pre-campaign', 'Post-campaign'], patch_artist=True,
                  boxprops=dict(alpha=0.6),
                  medianprops=dict(color='#1A3A5C', linewidth=2.5))
bp['boxes'][0].set_facecolor('#D97706')
bp['boxes'][1].set_facecolor('#0D7377')
ax2.set_ylabel('Weekly revenue (£k)')
ax2.set_title(f't = {t_stat2:.2f},  p < 0.001  →  Significant lift')
ax2.grid(axis='y', alpha=0.3)

plt.suptitle('Worked Example 2: Campaign effect on weekly revenue', fontsize=13)
plt.tight_layout(); plt.show()

## Your Turn 2 — Does the campaign effect differ by store size?

Split the post-campaign data into **Large** vs **Small** stores.  
Test whether large stores saw a bigger revenue lift than small stores.

**Refer to Slide 12** for the two-sample structure.


In [ ]:
# ── YOUR TURN 2 ─────────────────────────────────────────────────────────────
# STEP 1: State hypotheses
# H₀: ?
# Hₐ: ?

post_large = df[(df['period']=='Post') & (df['store_size']=='Large')]['weekly_revenue_k']
post_small = df[(df['period']=='Post') & (df['store_size']=='Small')]['weekly_revenue_k']

# STEP 2: Descriptives
# YOUR CODE HERE

# STEP 3: Two-sample t-test
# t_yt2, p_yt2 = stats.ttest_ind(post_large, post_small, equal_var=False)

# STEP 4: Decision and interpretation
# YOUR CODE HERE


*Interpretation: ...*

> ...

---
# Exercise 3 — Confidence Interval for the Campaign Effect

**Business question**: We know the campaign worked — but by how much, and how precisely?

> **Refer to Slides 11–12** for the decision-rule logic.  
> **Formula**: `CI = (x-bar_post − x-bar_pre) ± 2 × SE_diff`


## Worked Example 3 — 95% CI for the revenue lift (post vs pre)

In [ ]:
# ── WORKED EXAMPLE 3 ────────────────────────────────────────────────────────
# STEP 1: Hypotheses
# H₀: μ_post − μ_pre = 0  (no lift)
# Hₐ: difference ≠ 0
# α = 0.05

pre  = df[df['period']=='Pre']['weekly_revenue_k']
post = df[df['period']=='Post']['weekly_revenue_k']

# STEP 2: Compute difference and SE
diff    = post.mean() - pre.mean()
se_diff = np.sqrt(pre.var()/len(pre) + post.var()/len(post))

print("─── Step 2: Descriptives ───────────────────────────────")
print(f"  Mean difference: £{diff:.2f}k/week")
print(f"  SE of diff:      £{se_diff:.3f}k")

# STEP 3: t-test + CI
t_stat3, p_value3 = stats.ttest_ind(post, pre, equal_var=False)
ci_low  = diff - 2 * se_diff
ci_high = diff + 2 * se_diff

print(f"\n─── Step 3: t-test & 95% CI ────────────────────────────")
print(f"  t-statistic: {t_stat3:.3f}")
print(f"  p-value:     {p_value3:.6f}")
print(f"  95% CI:      [£{ci_low:.2f}k,  £{ci_high:.2f}k]")
print(f"  Includes 0?  {ci_low < 0 < ci_high}")

# STEP 4: Decision
print(f"\n─── Step 4: Decision ───────────────────────────────────")
print(f"  CI does NOT include zero  →  REJECT H₀")
print(f"  The campaign increased revenue by between £{ci_low:.1f}k and £{ci_high:.1f}k per")
print(f"  store per week (95% confidence).")
print(f"\n  Business interpretation: a £{ci_low:.1f}k–£{ci_high:.1f}k weekly lift across 120")
print(f"  stores = £{ci_low*120:.0f}k–£{ci_high*120:.0f}k total weekly revenue gain.")

## Your Turn 3 — CI for the revenue gap between North and East regions

Do North and East region stores earn different amounts on average?  
Compute the difference, run the test, and build the 95% CI.  
Does the CI include zero?


In [ ]:
# ── YOUR TURN 3 ─────────────────────────────────────────────────────────────
# STEP 1: Hypotheses
# H₀: ?
# Hₐ: ?

north = df[df['region']=='North']['weekly_revenue_k']
east  = df[df['region']=='East']['weekly_revenue_k']

# STEP 2: Difference and SE
# diff_ne    = north.mean() - east.mean()
# se_diff_ne = np.sqrt(north.var()/len(north) + east.var()/len(east))

# STEP 3: t-test and 95% CI
# ci_low  = diff_ne - 2 * se_diff_ne
# ci_high = diff_ne + 2 * se_diff_ne

# STEP 4: Does the CI include zero? Business interpretation?
# YOUR CODE HERE


*Interpretation: ...*

> ...

---
# Exercise 4 — Multiple Testing & Bonferroni Correction

**Business question**: Which pairs of regions have significantly different weekly revenue? Use the Bonferroni correction.

> **Refer to Slide 13 (Worked Example C)** — this exercise replicates it exactly.  
> **Bonferroni formula**: `threshold = α / number of tests = 0.05 / 6 = 0.0083`


## Worked Example 4 — All pairwise region comparisons (revenue)

In [ ]:
# ── WORKED EXAMPLE 4 ────────────────────────────────────────────────────────
regions    = sorted(df['region'].unique())
pairs      = list(combinations(regions, 2))
alpha_naive = 0.05
alpha_bonf  = alpha_naive / len(pairs)

print(f"Regions: {regions}")
print(f"Pairs: {len(pairs)}   Bonferroni threshold: {alpha_bonf:.4f}\n")
print(f"{'Pair':<22} | {'t':>7} | {'p-value':>9} | {'Naive (5%)':>11} | {'Bonferroni':>11}")
print("─" * 70)

results = []
for r1, r2 in pairs:
    g1 = df[df['region']==r1]['weekly_revenue_k']
    g2 = df[df['region']==r2]['weekly_revenue_k']
    t, p = stats.ttest_ind(g1, g2, equal_var=False)
    naive = 'Reject *' if p < alpha_naive else 'retain'
    bonf  = 'Reject *' if p < alpha_bonf  else 'RETAIN ←'
    results.append((f"{r1} vs {r2}", t, p, naive, bonf))
    print(f"{r1+' vs '+r2:<22} | {t:>7.2f} | {p:>9.4f} | {naive:>11} | {bonf:>11}")

n_naive = sum(1 for r in results if 'Reject' in r[3])
n_bonf  = sum(1 for r in results if 'Reject' in r[4])
print(f"\nRejections — Naive: {n_naive}/{len(pairs)}   Bonferroni: {n_bonf}/{len(pairs)}")
print(f"→ {n_naive-n_bonf} pair(s) flagged by naive 5% are likely false positives.")

## Your Turn 4 — Same analysis for store-size revenue comparisons

Run all **3 pairwise t-tests** on `weekly_revenue_k` across store sizes (Small, Medium, Large).  
Apply Bonferroni (3 pairs → threshold = 0.05/3 = 0.0167).  
Which size differences are genuine vs potentially spurious?


In [ ]:
# ── YOUR TURN 4 ─────────────────────────────────────────────────────────────
store_sizes = sorted(df['store_size'].unique())
size_pairs  = list(combinations(store_sizes, 2))
alpha_bonf_sizes = 0.05 / len(size_pairs)

print(f"Store sizes: {store_sizes}")
print(f"Pairs: {len(size_pairs)}   Bonferroni threshold: {alpha_bonf_sizes:.4f}\n")

# Hint: same structure as Worked Example 4 — loop over size_pairs
# for s1, s2 in size_pairs:
#     g1 = df[df['store_size']==s1]['weekly_revenue_k']
#     g2 = df[df['store_size']==s2]['weekly_revenue_k']
#     ...

# YOUR CODE HERE


*Which size differences survive Bonferroni? What does this suggest about store strategy?*

> ...

---
# Exercise 5 (Challenge) — Statistical vs Practical Significance

**Business question**: The campaign is statistically significant. But is a £2.9k/week lift actually meaningful for the business? Does it vary by store size?

> **Refer to Slide 10** — same concept as Scenario A vs B (same gap, different t).


## Worked Example 5 — Campaign effect: small vs large sample

In [ ]:
# ── WORKED EXAMPLE 5 ────────────────────────────────────────────────────────
# Compare using 15 stores vs all 120 stores — same true lift, different n
np.random.seed(42)
sample_stores = np.random.choice(df['store_id'].unique(), 15, replace=False)

pre_full   = df[df['period']=='Pre']['weekly_revenue_k']
post_full  = df[df['period']=='Post']['weekly_revenue_k']
pre_small  = df[(df['period']=='Pre')  & (df['store_id'].isin(sample_stores))]['weekly_revenue_k']
post_small = df[(df['period']=='Post') & (df['store_id'].isin(sample_stores))]['weekly_revenue_k']

print(f"{'Dataset':<14} | {'n/group':>7} | {'Mean diff':>10} | {'SE':>6} | {'t':>7} | {'p':>8} | Reject?")
print("─" * 68)
for label, g1, g2 in [
    ('Small (15 stores)', post_small, pre_small),
    ('Full (120 stores)', post_full,  pre_full),
]:
    diff = g1.mean() - g2.mean()
    se   = np.sqrt(g1.var()/len(g1) + g2.var()/len(g2))
    t, p = stats.ttest_ind(g1, g2, equal_var=False)
    print(f"{label:<14} | {len(g1):>7} | £{diff:>8.2f}k | {se:>6.2f} | {t:>7.2f} | {p:>8.4f} | {'YES' if abs(t)>2 else 'no'}")

print("\nSame direction of effect — the mean diff barely changes.")
print("What changes: SE shrinks as n grows → t grows → p shrinks.")
print("Large n gives the statistical power to detect a real (if modest) effect.")

## Your Turn 5 — Is the campaign effect practically meaningful by store size?

Separately test the campaign effect (post vs pre) for **Small**, **Medium**, and **Large** stores.

Fill in this table and reflect on whether a significant result is also a *meaningful* business result:

| Store size | Pre mean | Post mean | Lift (£k) | t-stat | p-value | Significant? |
|------------|----------|-----------|-----------|--------|---------|-------------|
| Small      | ?        | ?         | ?         | ?      | ?       | ?           |
| Medium     | ?        | ?         | ?         | ?      | ?       | ?           |
| Large      | ?        | ?         | ?         | ?      | ?       | ?           |


In [ ]:
# ── YOUR TURN 5 ─────────────────────────────────────────────────────────────
for size in ['Small', 'Medium', 'Large']:
    pre_s  = df[(df['period']=='Pre')  & (df['store_size']==size)]['weekly_revenue_k']
    post_s = df[(df['period']=='Post') & (df['store_size']==size)]['weekly_revenue_k']
    
    # YOUR CODE: compute diff, t, p for each size
    # t_s, p_s = stats.ttest_ind(post_s, pre_s, equal_var=False)
    # print(f"{size}: lift=£{post_s.mean()-pre_s.mean():.2f}k, t={t_s:.2f}, p={p_s:.4f}")
    pass


*Reflection: For which store sizes is the lift statistically significant? For which is it practically meaningful to the business?*

> ...

---
## Quick Reference

| Task | Python command |
|------|---------------|
| Load dataset | `df = pd.read_csv('retail_sales_data.csv')` |
| Filter period | `df[df['period'] == 'Pre']['weekly_revenue_k']` |
| Filter size | `df[df['store_size'] == 'Large']['weekly_revenue_k']` |
| One-sample test | `stats.ttest_1samp(data, popmean=target)` |
| Two-sample test | `stats.ttest_ind(group_a, group_b, equal_var=False)` |
| 95% CI | `diff ± 2 × SE_diff` |
| Bonferroni | `0.05 / number_of_tests` |

**Decision rule**: `|t| > 2` (or `p < 0.05`) → reject H₀ at 5% significance level.
